# 🏥 Bupa 국제 의료보험 RAG 챗봇 (LangGraph v2)

### 변경 사항 (v1 → v2)
| 구성 요소 | v1 | v2 |
|---|---|---|
| LLM | Claude Sonnet | `gpt-4.1-mini` |
| 임베딩 | text-embedding-3-small (영어 위주) | `BAAI/bge-m3` (100개 언어) |
| 아키텍처 | BupaRAGChatbot 클래스 | **LangGraph StateGraph** |
| 대화 메모리 | 수동 리스트 | `MemorySaver` (thread_id 기반) |

### LangGraph 플로우
```
START → [detect_plan] → [retrieve] → [generate] → END
           플랜 감지       메타필터 검색    LLM 답변
           모듈 추적       bge-m3 검색    gpt-4.1-mini
```

### 필수 준비
- `.env` 파일: `OPENAI_API_KEY=sk-...`
- 6개 Bupa PDF 파일 동일 폴더
- bge-m3 초회 다운로드 ~2.2GB (이후 로컬 캐시)

## 0. 패키지 설치

In [1]:
# # 처음 실행 시에만 주석 해제
# !pip install \
#   langgraph langchain langchain-community langchain-openai langchain-chroma \
#   langchain-huggingface sentence-transformers \
#   pdfplumber pymupdf python-dotenv chromadb openai -q

## 1. 환경 설정

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
openai_key = os.getenv('OPENAI_API_KEY')
print(f"OpenAI API Key: {'✅ 로드 성공' if openai_key else '❌ 실패 → .env 확인'}")

OpenAI API Key: ✅ 로드 성공


## 2. 전처리한 chromaDB 불러오기

In [3]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print('⏳ bge-m3 로딩 및 VRAM 6GB 최적화 중...')

# 1. 모델 설정
model_name = 'BAAI/bge-m3'
model_kwargs = {
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'model_kwargs': {'torch_dtype': torch.float16} # VRAM 절약 핵심
}
encode_kwargs = {'normalize_embeddings': True, 'batch_size': 8}

# 2. 임베딩 모델 객체 생성
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# 3. [에러 해결 포인트] Pydantic의 감시를 피해 내부 속성에 직접 접근
# 'client'라는 필드가 없다고 나오면 __dict__를 통해 강제로 접근합니다.
try:
    if hasattr(embedding_model, 'client'):
        target = embedding_model.client
    else:
        # Pydantic 내부 구조에서 실제 모델 객체 찾아내기
        target = embedding_model.__dict__.get('client') or embedding_model.__dict__.get('_client')
    
    if target:
        target.max_seq_length = 768 # 8192에서 768로 줄여서 메모리 확보
        print(f"✅ max_seq_length가 {target.max_seq_length}로 최적화되었습니다.")
except Exception as e:
    print(f"⚠️ 최적화 설정 중 사소한 문제 발생(무시 가능): {e}")

# 4. 벡터 DB 로드
CHROMA_DIR = './chroma_bupa_preprocessed'
vectorstore = Chroma(
    collection_name='bupa_preprocessed',
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR
)
print(f'✅ DB 로드 완료: {vectorstore._collection.count()}개 벡터')

c:\Users\kwonm\anaconda3\envs\nlp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ bge-m3 로딩 및 VRAM 6GB 최적화 중...


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 358.32it/s]


✅ max_seq_length가 768로 최적화되었습니다.
✅ DB 로드 완료: 864개 벡터


## 3. LangGraph State 정의

State는 그래프의 모든 노드가 공유하는 데이터 구조입니다.
각 노드는 State의 일부를 수정하여 반환하고, LangGraph가 자동으로 병합합니다.

```
BupaState
├── messages          ← add_messages 리듀서: HumanMessage/AIMessage 자동 누적
├── plan_tier         ← 감지된 플랜 티어 (None이면 detect_plan이 채움)
├── ihhp_modules      ← IHHP 모듈 목록 (대화 중 점진적 누적)
├── retrieved_docs    ← 이번 턴 검색 결과 (매 턴 교체)
└── current_question  ← 현재 처리 중인 질문 (노드 간 전달)
```

In [4]:
from typing import Annotated, Optional, TypedDict, Dict, Any
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class InsuranceState(TypedDict):
    # 1. 대화 기록 (누적)
    messages: Annotated[list[BaseMessage], add_messages]
    # 2. 감지된 플랜 정보 (기억 유지)
    plan_tier: Optional[str]
    ihhp_modules: list[str]
    # 3. 질문 분석 결과 (매 턴 갱신)
    normalized_query: Dict[str, Any]
    # 4. 검색된 문서들
    retrieved_docs: list[str]
    # 5. 현재 처리 중인 질문
    current_question: str

In [5]:
import json, re
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.retrievers import BM25Retriever

class InsuranceRAGFactory:
    def __init__(self, vectorstore, system_prompt, model_name="gpt-4o-mini"):
        self.vectorstore = vectorstore
        self.system_prompt = system_prompt
        # 분석용 LLM (정밀도 우선)
        self.analyzer_llm = ChatOpenAI(model=model_name, temperature=0)
        # 답변 생성용 LLM (창의성/공감 약간 가미)
        self.chat_llm = ChatOpenAI(model=model_name, temperature=0.1)

    # ────────────── [Node 1: Analyze] ──────────────
    def analyze_node(self, state: InsuranceState):
        question = state['messages'][-1].content
        
        # 질문에서 언어, 의도, 플랜, 영어 쿼리를 한 번에 추출
        prompt = f"""You are an insurance query analyzer.
        Extract info and return STRICT JSON:
        {{
          "language": "ko|en|zh|ja...",
          "section_type": "benefit_table|exclusion|claim_process|pre_auth|etc",
          "plan_tier": "Elite|Premier|Select|MajorMedical|IHHP|None",
          "english_query": "<search query here>"
        }}
        
        Rules for section_type:
        - benefit_table: 보장 여부, 보장 한도, 보장 금액 관련 질문 → "보장돼?", "얼마나 나와?", "커버돼?"
        - exclusion: 명시적으로 제외/미보장 항목을 묻는 경우 → "안 되는게 뭐야?", "제외 항목이 뭐야?"
        - claim_process: 청구 방법, 서류, 절차 관련 → "어떻게 청구해?", "서류가 뭐야?"
        - pre_auth: 사전승인 관련 → "사전승인 필요해?", "미리 허가 받아야 해?"
        - 애매하면 benefit_table로 분류하세요
        
        Rules for english_query:
        - Use EXACT terminology from insurance benefit documents
        - Good: "prescribed medicines dressings benefit limit GBP"
        - Bad: "coverage for cold medication"
        
        Question: {question}"""
        
        raw_res = self.analyzer_llm.invoke(prompt).content
        # JSON만 추출하는 안전장치
        json_str = re.search(r'\{.*\}', raw_res, re.DOTALL).group(0)
        analysis = json.loads(json_str)
        
        # 새로운 플랜이 감지되면 업데이트, 아니면 기존 플랜 유지
        detected_plan = analysis.get('plan_tier')
        final_plan = detected_plan if detected_plan != "None" else state.get('plan_tier')

        return {
            "normalized_query": analysis,
            "plan_tier": final_plan,
            "current_question": question
        }

        # ────────────── [Node 2: Retrieve] ──────────────
    # retrieve_node에서 RRF 부분을 이렇게 교체
    def retrieve_node(self, state: InsuranceState):
        query = state['normalized_query'].get('english_query', state['current_question'])
        plan = state.get('plan_tier')
        section = state['normalized_query'].get('section_type')
    
        filters = []
        if plan and plan != "None":
            filters.append({"plan_tier": plan})
        if section:
            filters.append({"section_type": section})
        search_filter = {"$and": filters} if len(filters) > 1 else (filters[0] if filters else None)
    
        docs = self.vectorstore.similarity_search(query, k=8, filter=search_filter)
        if not docs:
            docs = self.vectorstore.similarity_search(query, k=8)
    
        formatted = [f"[p.{d.metadata.get('page', '?')}] {d.page_content}" for d in docs]
        return {"retrieved_docs": formatted}
    
    # ────────────── [Node 3: Generate] ──────────────
    def generate_node(self, state: InsuranceState):
        context = "\n\n".join(state['retrieved_docs'])

        # 감지된 언어로 답변하도록 지시 추가
        language = state.get('normalized_query', {}).get('language', 'ko')
        language_map = {
            'ko': '한국어', 'en': 'English', 'ja': '日本語',
            'zh': '中文', 'fr': 'Français', 'de': 'Deutsch', 'es': 'Español'
        }
        answer_language = language_map.get(language, '한국어')

        messages = [
            SystemMessage(content=self.system_prompt + f"\n\n[언어 규칙] 반드시 {answer_language}로 답변하세요."),
            *state['messages'][:-1],
            HumanMessage(content=f"[Documents]\n{context}\n\n[User Question]\n{state['current_question']}")
        ]

        response = self.chat_llm.invoke(messages)
        return {"messages": [response]}

    # ────────────── [Graph Compilation] ──────────────
    def build(self):
        builder = StateGraph(InsuranceState)
        builder.add_node("analyze", self.analyze_node)
        builder.add_node("retrieve", self.retrieve_node)
        builder.add_node("generate", self.generate_node)
        
        builder.add_edge(START, "analyze")
        builder.add_edge("analyze", "retrieve")
        builder.add_edge("retrieve", "generate")
        builder.add_edge("generate", END)
        
        return builder.compile(checkpointer=MemorySaver())

In [6]:
# 2. Bupa 전용 시스템 프롬프트 작성
BUPA_PROMPT = """당신은 Bupa 국제 의료보험 전문 상담사입니다.

[중요: 플랜별 판단 로직]
1. Premier, Elite, Select, Major Medical 플랜:
   - 이 플랜들은 모듈형이 아닙니다.
   - [Documents]에 'Paid in full', 'Full refund', 'will be paid'라고 적혀 있으면 무조건 "100% 보장"입니다.
   - 절대로 "모듈이 필요하다"거나 "보장되지 않는다"고 하지 마세요.

2. IHHP (International Health Insurance Plan) 플랜:
   - 이 플랜만 유일하게 모듈형입니다.
   - 문서에 '※ 모듈형 표' 또는 '[일부 모듈 가입 시 보장]' 태그가 있으면 "특정 모듈 가입 시 보장"으로 안내하세요.
   - Not Covered만 보고 전체 미보장으로 판단하지 마세요. 다른 모듈 컬럼에 Covered가 있을 수 있습니다.

[답변 구성 원칙]
- 보장 여부, 전제조건, 제외 항목, 한도를 모두 포함하되, 항목 제목을 달지 말고 자연스러운 대화체 문장으로 이어서 작성하세요.
- 반드시 포함할 내용: 보장 여부 → 필요한 조건 → 안 되는 케이스 → 금액 한도 → 출처
- 좋은 예시:
  "Premier 플랜에서 암 치료는 100% 보장됩니다. 단, 치료 전에 Bupa에 사전승인(Pre-authorization: 치료 전 보험사에 미리 승인을 받는 절차)을 받아야 하며, 보험 가입 전부터 있던 기존 질환(Pre-existing condition)으로 인한 치료는 보장에서 제외됩니다. 연간 한도는 GBP 1,500,000이며, 출처는 p.18입니다."
- 나쁜 예시: "보장됩니다. 전제조건: ... 제외 항목: ... 한도: ..."처럼 항목을 나열하지 마세요.

[전문 용어 처리 원칙]
- 답변에 아래 용어가 등장하면, 처음 사용 시 괄호로 간단한 설명을 덧붙이세요.
  예) Co-insurance(공동부담금: 보험사와 가입자가 비용을 나눠 부담하는 비율)
- 반복 등장 시에는 설명 없이 용어만 사용해도 됩니다.
- 아래는 자주 나오는 용어 목록입니다:
  · Deductible / Excess: 보험금 지급 전 가입자가 먼저 부담하는 금액 (자기부담금)
  · Co-insurance: 자기부담금 이후 남은 비용을 보험사와 가입자가 나누는 비율
  · Pre-authorization / Prior approval: 치료 전 보험사에 미리 승인을 받는 절차
  · In-patient: 1박 이상 입원한 경우
  · Out-patient: 입원 없이 외래로 진료받는 경우
  · Day-patient: 당일 입원 후 퇴원 (1박 미만)
  · Paid in full: 가입자 부담 없이 100% 보장
  · Annual maximum / Benefit limit: 1년간 보험사가 지급하는 최대 금액
  · Waiting period: 보험 가입 후 보장이 시작되기까지의 대기 기간
  · Pre-existing condition: 보험 가입 전부터 존재했던 질환
  · Moratorium: 일정 기간 동안 기존 질환 관련 보장을 유예하는 조건
  · OTC (Over-the-counter): 처방전 없이 약국에서 구매 가능한 일반의약품

[불명확한 질문 처리 원칙]
- 아래 경우에만 되묻고, 나머지는 정상 답변하세요.

CASE 1 - 지시어만 있는 경우 ("이거", "그거", "이 치료", "이 항목" 등 대상이 특정되지 않은 경우):
  반드시 "어떤 치료/항목에 대해 문의하시는 건가요?" 라고 되물으세요.
  절대 추측해서 답하지 마세요. 예시처럼 답변해도 되겠다고 판단해도 되물어야 합니다.

CASE 2 - 구체적 항목이 있지만 플랜이 없는 경우 ("치료비 보장돼요?", "약값 나와요?" 등):
  "어떤 플랜에 가입하셨나요?" 라고 확인하세요.
  단, 플랜 비교 질문("A랑 B 플랜 차이가 뭐예요?")은 예외로 양쪽 다 설명하세요.

CASE 3 - 대화 기록이 없는데 이전 대화 참조 ("아까", "방금", "그 한도" 등):
  "이전 대화 내용을 찾을 수 없어요. 어떤 항목의 한도를 말씀하시는 건가요?" 라고 되물으세요.

[주의]
- 전제조건이나 예외사항이 문서에 있다면 절대 생략하지 마세요.
- 단순히 "보장됩니다"로만 끝내지 말고, 사용자가 실제로 청구 가능한지 판단할 수 있도록 조건을 함께 안내하세요.
- 문서에 없는 내용은 추측하지 말고 "문서에 명시되지 않았습니다"라고 하세요.
"""

# 3. 팩토리 가동! (Bupa 봇 찍어내기)
bupa_factory = InsuranceRAGFactory(
    vectorstore=vectorstore, 
    system_prompt=BUPA_PROMPT
)
bupa_bot = bupa_factory.build()

# 4. 채팅 실행
config = {"configurable": {"thread_id": "user_111"}}
result = bupa_bot.invoke({"messages": [HumanMessage(content="Premier 플랜 암 치료 보장돼?")]}, config)
print(result['messages'][-1].content)

Premier 플랜에서 암 치료는 100% 보장됩니다. 단, 치료를 진행하기 전에 Bupa에 사전승인(Pre-authorization)을 받아야 하며, 사전승인이 없으면 보장이 이루어지지 않습니다. 암 치료에는 진단 후 치료 계획 수립 및 실행에 관련된 비용, 검사, 진단 영상, 상담 및 처방된 약품이 포함됩니다. 연간 한도는 GBP 1,500,000이며, 출처는 p.25입니다.


### 테스트

In [7]:
config = {"configurable": {"thread_id": "test_ihhp_new"}}
result = bupa_bot.invoke(
    {"messages": [HumanMessage(content="IHHP 플랜 치과 치료 보장돼?")]}, 
    config
)
print(result['messages'][-1].content)

IHHP 플랜에서 치과 치료는 특정 모듈 가입 시 보장됩니다. 예를 들어, Module 4A와 Module 4B에 따라 치과 치료 항목이 다르게 보장되며, 각 모듈에 따라 보장 한도가 설정되어 있습니다. 예를 들어, 치과 검진은 Module 4A에서 최대 EUR 30(GBP 25)까지 80% 보장되며, Module 4B에서는 최대 EUR 50(GBP 40)까지 80% 보장됩니다. 

또한, 치료를 받기 전에 해당 모듈에 가입되어 있어야 하며, 가입하지 않은 경우에는 보장되지 않습니다. 따라서, 어떤 모듈에 가입하셨는지 확인하시고, 그에 따라 보장 여부를 판단하셔야 합니다. 출처는 p.16입니다.


In [8]:
config = {"configurable": {"thread_id": "user_123"}}
result = bupa_bot.invoke({"messages": [HumanMessage(content="Premier 플랜인데, 감기약 사먹은 것도 보장돼?")]}, config)
print(result['messages'][-1].content)

감기약은 일반의약품(OTC)으로 분류되기 때문에 Premier 플랜에서는 보장되지 않습니다. 이 플랜은 처방전 없이 구매 가능한 약품에 대한 보장을 제공하지 않으며, 보장되는 항목은 의사가 처방한 약품과 드레싱에 한정됩니다. 따라서 감기약은 보장 대상이 아닙니다. 문서에 명시된 내용입니다.


### Chat 함수 정의

In [9]:
def chat(question: str, thread_id: str = 'default') -> str:
    config = {'configurable': {'thread_id': thread_id}}
    
    # bupa_bot의 현재 상태를 가져옴
    state_snapshot = bupa_bot.get_state(config)
    existing_plan = state_snapshot.values.get('plan_tier') if state_snapshot.values else None
    
    # bupa_bot에게 일을 시킴 (invoke)
    result = bupa_bot.invoke(
        {
            "messages": [HumanMessage(content=question)],
            "plan_tier": existing_plan
        }, 
        config=config
    )
    
    return result['messages'][-1].content

# 3. 상태 확인용 대시보드 함수
def get_state(thread_id: str = 'default') -> dict:
    config = {'configurable': {'thread_id': thread_id}}
    state_snapshot = bupa_bot.get_state(config) # bupa_bot의 상태 조회
    
    if not state_snapshot.values:
        return {"status": "데이터 없음"}
    
    return {
        "plan_tier": state_snapshot.values.get("plan_tier"),
        "turns": len([m for m in state_snapshot.values.get("messages", []) if isinstance(m, HumanMessage)])
    }

### 12-B. 멀티턴 (GHP Select)

In [10]:
T = 'ghp_select'
turns = [
    '안녕하세요, Bupa Select 플랜 가입자입니다.',
    '입원 시 1일 최대 보장 금액이 얼마인가요?',
    '외래 진료는 어떻게 되나요?',
    '긴급 치과 치료도 포함인가요?',
]

for i, q in enumerate(turns, 1):
    print(f'\n[Turn {i}] 👤 {q}')
    print(f'🤖 {chat(q, T)}')
    print(f'📊 {get_state(T)}')
    print('-' * 65)


[Turn 1] 👤 안녕하세요, Bupa Select 플랜 가입자입니다.
🤖 안녕하세요! Bupa Select 플랜에 가입하셨군요. 어떤 치료나 항목에 대해 문의하시는 건가요?
📊 {'plan_tier': 'Select', 'turns': 1}
-----------------------------------------------------------------

[Turn 2] 👤 입원 시 1일 최대 보장 금액이 얼마인가요?
🤖 Bupa Select 플랜에서는 입원 시 병원 숙소, 치료 및 관련 비용이 100% 보장됩니다. 단, 입원 기간이 5일 이상인 경우에는 Bupa에 사전승인(Pre-authorization)을 받아야 하며, 이 경우 진단서와 치료 계획을 제출해야 합니다. 연간 최대 보장 금액은 GBP 1,000,000이며, 출처는 p.18입니다.
📊 {'plan_tier': 'Select', 'turns': 2}
-----------------------------------------------------------------

[Turn 3] 👤 외래 진료는 어떻게 되나요?
🤖 Bupa Select 플랜에서는 외래 진료에 대한 비용이 연간 최대 GBP 7,500까지 100% 보장됩니다. 다만, 외래 진료를 받을 때는 15%의 공동부담금(Co-insurance)이 적용되며, 선택적으로 25%로 증가시킬 수 있습니다. 외래 진료는 의사나 전문의의 추천이 필요하며, 치료가 보험 약관에 포함되어 있어야 합니다. 또한, 외래 진료는 입원 치료와는 달리 사전승인(Pre-authorization)이 필요하지 않습니다. 출처는 p.18입니다.
📊 {'plan_tier': 'Select', 'turns': 3}
-----------------------------------------------------------------

[Turn 4] 👤 긴급 치과 치료도 포함인가요?
🤖 Bupa Select 플랜에서는 긴급 치과 치료가 보장됩니다. 하지만 이 치료는 심

### 12-C. 멀티턴 (IHHP 모듈형)

In [11]:
T = 'ihhp'
turns = [
    'IHHP 플랜 쓰고 있어요. 치과 치료 보장되나요?',
    '임플란트도 되나요?',
    '연간 치과 보장 한도가 얼마예요?',
]

for i, q in enumerate(turns, 1):
    print(f'\n[Turn {i}] 👤 {q}')
    print(f'🤖 {chat(q, T)}')
    print(f'📊 {get_state(T)}')
    print('-' * 65)


[Turn 1] 👤 IHHP 플랜 쓰고 있어요. 치과 치료 보장되나요?
🤖 IHHP 플랜에서 치과 치료는 특정 모듈 가입 시 보장됩니다. 예를 들어, Module 4A와 Module 4B에서 치과 치료 항목이 포함되어 있으며, 각 항목에 대해 80% 보장됩니다. 예를 들어, 치과 검진은 최대 EUR 30 (GBP 25 또는 USD 30)까지 보장되며, 치아 청소는 최대 EUR 50 (GBP 30 또는 USD 50)까지 보장됩니다. 

하지만, 만약 해당 모듈에 가입하지 않았다면, 치과 치료는 보장되지 않습니다. 또한, 각 항목마다 연간 한도가 있으니, 치료를 받기 전에 어떤 모듈에 가입했는지 확인하는 것이 중요합니다. 출처는 p.16입니다.
📊 {'plan_tier': 'IHHP', 'turns': 1}
-----------------------------------------------------------------

[Turn 2] 👤 임플란트도 되나요?
🤖 IHHP 플랜에서 임플란트 치료는 특정 모듈 가입 시 보장됩니다. Module 4A와 Module 4B에서 임플란트 치료는 연간 최대 EUR 2,650 (GBP 2,000 또는 USD 2,650)까지 50% 보장됩니다. 하지만, 만약 해당 모듈에 가입하지 않았다면 임플란트 치료는 보장되지 않습니다. 따라서, 어떤 모듈에 가입했는지 확인하는 것이 중요합니다. 출처는 p.16입니다.
📊 {'plan_tier': 'IHHP', 'turns': 2}
-----------------------------------------------------------------

[Turn 3] 👤 연간 치과 보장 한도가 얼마예요?
🤖 IHHP 플랜에서 연간 치과 보장 한도는 모듈에 따라 다릅니다. Module 4A의 경우 연간 최대 EUR 5,000 (GBP 3,500 또는 USD 5,000)까지 보장되며, Module 4B의 경우 연간 최대 EUR 7,500 (GBP 5,000 또는 USD 7,500)까지 보장됩니

### 12-D. 다국어 테스트 (bge-m3 교차 언어 검증)

In [12]:
# 동일 의미, 다른 언어 → bge-m3가 영어 PDF에서 모두 검색 성공해야 함
tests = [
    ('한국어', 'ko', 'Select 플랜에서 연간 보장 한도가 얼마인가요?'),
    ('English', 'en', 'What is the annual benefit maximum for the Select plan?'),
    ('日本語', 'ja', 'セレクトプランの年間保障限度額はいくらですか?'),
]

print('🌐 다국어 교차 언어 검색 테스트')
print('=' * 65)

for lang, tid, q in tests:
    print(f'\n[{lang}] 👤 {q}')
    ans = chat(q, thread_id=f'ml_{tid}')
    print(f'🤖 {ans[:400]}...' if len(ans) > 400 else f'🤖 {ans}')
    print('-' * 65)

🌐 다국어 교차 언어 검색 테스트

[한국어] 👤 Select 플랜에서 연간 보장 한도가 얼마인가요?
🤖 Select 플랜의 연간 보장 한도는 GBP 1,000,000, EUR 1,250,000, 또는 USD 1,700,000입니다. 이 한도는 모든 혜택에 적용되며, 모든 혜택이 연간 최대 한도에 기여하게 됩니다. 출처는 p.18입니다.
-----------------------------------------------------------------

[English] 👤 What is the annual benefit maximum for the Select plan?
🤖 The annual benefit maximum for the Select plan is GBP 1,000,000, EUR 1,250,000, or USD 1,700,000. This limit applies to the total amount that Bupa will pay for all benefits in each policy year. Once this limit is reached, no further benefits will be available until the policy is renewed. You can find this information on page 18 of the documents.
-----------------------------------------------------------------

[日本語] 👤 セレクトプランの年間保障限度額はいくらですか?
🤖 セレクトプランの年間保障限度額は、GBP 1,000,000、EUR 1,250,000、またはUSD 1,700,000です。この限度額は、すべての給付に適用され、年間のポリシー最大限度額として設定されています。出典はp.18です。
-----------------------------------------------------------------


In [13]:
# 애매한 질문 테스트
ambiguous_cases = [
    ("vague_plan",    "보험 되나요?"),                        # 플랜 언급 없음
    ("vague_item",    "Premier 플랜인데 이거 보장돼요?"),      # '이거'가 뭔지 모름
    ("vague_multi",   "치료비랑 약값이랑 입원비 다 나와요?"),  # 여러 항목 동시
    ("vague_context", "아까 그 한도가 얼마라고요?"),           # 대화 맥락 의존
]

print("=" * 65)
print("🧪 애매한 질문 테스트")
print("=" * 65)

for tid, q in ambiguous_cases:
    print(f"\n❓ [{tid}] {q}")
    print(f"🤖 {chat(q, thread_id=tid)}")
    print(f"📊 {get_state(tid)}")
    print("-" * 65)

🧪 애매한 질문 테스트

❓ [vague_plan] 보험 되나요?
🤖 어떤 치료/항목에 대해 문의하시는 건가요?
📊 {'plan_tier': None, 'turns': 1}
-----------------------------------------------------------------

❓ [vague_item] Premier 플랜인데 이거 보장돼요?
🤖 어떤 치료/항목에 대해 문의하시는 건가요?
📊 {'plan_tier': 'Premier', 'turns': 1}
-----------------------------------------------------------------

❓ [vague_multi] 치료비랑 약값이랑 입원비 다 나와요?
🤖 어떤 플랜에 가입하셨나요?
📊 {'plan_tier': None, 'turns': 1}
-----------------------------------------------------------------

❓ [vague_context] 아까 그 한도가 얼마라고요?
🤖 이전 대화 내용을 찾을 수 없어요. 어떤 항목의 한도를 말씀하시는 건가요?
📊 {'plan_tier': None, 'turns': 1}
-----------------------------------------------------------------


In [15]:
import numpy as np

def show_retrieval_scores(question: str, plan_tier: str = None, k: int = 5):
    from langchain_openai import ChatOpenAI
    import json, re

    analyzer = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = f"""You are an insurance query analyzer.
    Extract info from the question and return STRICT JSON:
    {{
      "language": "ko|en|zh|ja...",
      "section_type": "benefit_table|exclusion|claim_process|pre_auth|etc",
      "plan_tier": "Elite|Premier|Select|MajorMedical|IHHP|None",
      "english_query": "<search query here>"
    }}

    Rules for section_type:
    - benefit_table: 보장 여부, 보장 한도, 보장 금액 관련 질문 → "보장돼?", "얼마나 나와?", "커버돼?"
    - exclusion: 명시적으로 제외/미보장 항목을 묻는 경우 → "안 되는게 뭐야?", "제외 항목이 뭐야?"
    - claim_process: 청구 방법, 서류, 절차 관련 → "어떻게 청구해?", "서류가 뭐야?"
    - pre_auth: 사전승인 관련 → "사전승인 필요해?", "미리 허가 받아야 해?"
    - 애매하면 benefit_table로 분류하세요
    
    Rules for english_query:
    - Use EXACT terminology from insurance benefit documents
    - Good: "prescribed medicines dressings benefit limit GBP"
    - Bad: "coverage for cold medication"

     Question: {question}"""

    raw = analyzer.invoke(prompt).content

    json_str = re.search(r'\{.*\}', raw, re.DOTALL).group(0)
    analysis = json.loads(json_str)
    english_query = analysis.get("english_query", question)

    filters = []
    if plan_tier and plan_tier != "None":
        filters.append({"plan_tier": plan_tier})
    if analysis.get("section_type"):
        filters.append({"section_type": analysis["section_type"]})
    search_filter = {"$and": filters} if len(filters) > 1 else (filters[0] if filters else None)

    results = vectorstore.similarity_search_with_score(english_query, k=k, filter=search_filter)
    if not results:
        print(f"⚠️  메타필터 결과 없음 → 필터 제거 후 재검색")
        results = vectorstore.similarity_search_with_score(english_query, k=k)
    if not results:
        print("❌ 검색 결과가 없습니다.")
        return

    # L2 거리 → 코사인 유사도 변환 (normalize=True일 때 정확)
    def l2_to_cosine(l2):
        return round(1 - (l2 ** 2) / 2, 4)

    scores_l2  = [score for _, score in results]
    scores_cos = [l2_to_cosine(s) for s in scores_l2]
    max_cos, min_cos = max(scores_cos), min(scores_cos)

    print(f"\n{'='*65}")
    print(f"❓ 원본 질문  : {question}")
    print(f"🔍 영어 쿼리  : {english_query}")
    print(f"🏷️  섹션 타입  : {analysis.get('section_type')}")
    print(f"🔒 메타필터   : {search_filter}")
    print(f"{'='*65}")

    for i, ((doc, l2), cos) in enumerate(zip(results, scores_cos), 1):
        # 코사인 유사도 기준 바 (높을수록 유사)
        normalized = (cos - min_cos) / (max_cos - min_cos + 1e-9)
        bar = "█" * int(normalized * 20) + "░" * (20 - int(normalized * 20))

        # 구간별 판정
        if cos >= 0.80:
            grade = "🟢 좋음"
        elif cos >= 0.65:
            grade = "🟡 보통"
        else:
            grade = "🔴 낮음"

        print(f"\n[{i}위] 코사인 유사도: {cos:.4f}  [{bar}]  {grade}")
        print(f"     L2 거리: {l2:.4f}")
        print(f"     출처: {doc.metadata.get('source','?')} | p.{doc.metadata.get('page','?')} | {doc.metadata.get('section_type','?')}")
        print(f"     내용: {doc.page_content[:120].strip()}...")

    print(f"\n💡 코사인 유사도 범위: {min_cos:.4f} ~ {max_cos:.4f}  (높을수록 질문과 가까움)")
    print(f"   기준: 🟢 0.80+ 좋음 | 🟡 0.65~0.79 보통 | 🔴 0.65 미만 낮음")

# ── 테스트 ──
show_retrieval_scores("감기약 보장돼?", plan_tier="Premier")


❓ 원본 질문  : 감기약 보장돼?
🔍 영어 쿼리  : prescribed medicines dressings benefit limit GBP
🏷️  섹션 타입  : benefit_table
🔒 메타필터   : {'$and': [{'plan_tier': 'Premier'}, {'section_type': 'benefit_table'}]}

[1위] 코사인 유사도: 0.7415  [███████████████████░]  🟡 보통
     L2 거리: 0.7191
     출처: Bupa-Global-DAC-Premier-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf | p.19 | benefit_table
     내용: disease or illness, such as diabetes.
Paid in full* up to 4 visits
each policy year
Paid in full* up to 4 visits
each po...

[2위] 코사인 유사도: 0.7188  [████████████████░░░░]  🟡 보통
     L2 거리: 0.7499
     출처: Bupa-Global-DAC-Premier-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf | p.18 | benefit_table
     내용: TABLE OF BENEFITS
PREMIER HEALTH PLAN
BENEFIT AND EXPLANATION
LIMITS
ALL BENEFITS BELOW, EVEN THOSE PAID IN FULL WILL CO...

[3위] 코사인 유사도: 0.6351  [████░░░░░░░░░░░░░░░░]  🔴 낮음
     L2 거리: 0.8543
     출처: Bupa-Global-DAC-Premier-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf | p.18 | benefit_table
     내용: [Bupa GHP

In [ ]:
# p.19 청크 직접 조회
results = vectorstore.get(
    where={"$and": [{"plan_tier": "Premier"}, {"page": "19"}]}
)
for i, (doc, meta) in enumerate(zip(results['documents'], results['metadatas'])):
    print(f"\n[{i+1}] p.{meta.get('page')} | {meta.get('section_type')}")
    print(doc[:300])
    print("-" * 50)


[1] p.19 | benefit_table
BENEFIT AND EXPLANATION
LIMITS
SPECIALIST CONSULTATIONS AND DOCTOR'S FEES
Consultations with your specialist or doctor, for example to: 
receive or arrange treatment
follow up on treatment already received
receive routine baby/childhood check-ups
receive pre- and post-hospital consultations/treatmen
--------------------------------------------------

[2] p.19 | benefit_table
receive or arrange treatment
receive pre- and post-hospital treatment, or
diagnose your illness
PHYSIOTHERAPISTS, OSTEOPATHS AND CHIROPRACTORS
Consultations and treatment with physiotherapists, osteopaths, chiropractors for
physical therapies aimed at restoring your normal physical function.
OCCUPAT
--------------------------------------------------

[3] p.19 | benefit_table
disease or illness, such as diabetes.
Paid in full* up to 4 visits
each policy year
Paid in full* up to 4 visits
each policy year
PRESCRIBED MEDICINES AND DRESSINGS
Medicines and dressings prescribed by your medical pr

# 평가

In [ ]:
eval_dataset = [
    # ── Premier ──────────────────────────────────────────
    {
        "question": "Premier 플랜에서 처방약 한도가 얼마야?",
        "plan_tier": "Premier",
        "answer_page": "19",
        "answer_keyword": "PRESCRIBED MEDICINES AND DRESSINGS"
    },
    {
        "question": "Premier 플랜 전체 연간 한도는?",
        "plan_tier": "Premier",
        "answer_page": "18",
        "answer_keyword": "annual" 
    },
    {
        "question": "Premier 플랜에서 암 치료 보장돼?",
        "plan_tier": "Premier",
        "answer_page": "18",
        "answer_keyword": "cancer treatment"
    },
    {
        "question": "Premier 플랜 외래 진료 연간 한도가 얼마야?",
        "plan_tier": "Premier",
        "answer_page": "18",
        "answer_keyword": "OUT-PATIENT DAY TO DAY CARE"
    },
    {
        "question": "Premier 플랜에서 치과 치료 보장돼?",
        "plan_tier": "Premier",
        "answer_page": "20",
        "answer_keyword": "DENTAL TREATMENT"
    },
    {
        "question": "Premier 플랜에서 사전승인 필요한 항목이 뭐야?",
        "plan_tier": "Premier",
        "answer_page": "18",
        "answer_keyword": "MANDATORY PRE-AUTHORISATION"
    },
    {
        "question": "Premier 플랜 입원 시 병실 보장은?",
        "plan_tier": "Premier",
        "answer_page": "21",
        "answer_keyword": "HOSPITAL ACCOMMODATION"
    },
    {
        "question": "Premier 플랜 정신건강 상담 보장돼?",
        "plan_tier": "Premier",
        "answer_page": "19",
        "answer_keyword": "MENTAL HEALTH"
    },

    # ── Select ───────────────────────────────────────────
    {
        "question": "Select 플랜 연간 보장 한도가 얼마야?",
        "plan_tier": "Select",
        "answer_page": "18",
        "answer_keyword": "Overall annual policy maximum"
    },
    {
        "question": "Select 플랜 처방약 한도는?",
        "plan_tier": "Select",
        "answer_page": "19",
        "answer_keyword": "PRESCRIBED MEDICINES AND DRESSINGS"
    },
    {
        "question": "Select 플랜 외래 진료 co-insurance가 어떻게 돼?",
        "plan_tier": "Select",
        "answer_page": "18",
        "answer_keyword": "Co-insurance"
    },
    {
        "question": "Select 플랜에서 전문의 상담 횟수 한도는?",
        "plan_tier": "Select",
        "answer_page": "19",
        "answer_keyword": "SPECIALIST CONSULTATIONS"
    },

    # ── MajorMedical ─────────────────────────────────────
    {
        "question": "Major Medical 플랜 공제액(deductible)이 얼마야?",
        "plan_tier": "MajorMedical",
        "answer_page": "18",
        "answer_keyword": "DEDUCTIBLE"
    },
    {
        "question": "Major Medical 플랜 전체 연간 한도는?",
        "plan_tier": "MajorMedical",
        "answer_page": "18",
        "answer_keyword": "Overall annual policy maximum"
    },
    {
        "question": "Major Medical 플랜 입원 병실 보장은?",
        "plan_tier": "MajorMedical",
        "answer_page": "18",
        "answer_keyword": "HOSPITAL ACCOMMODATION"
    },

    # ── IHHP ─────────────────────────────────────────────
    {
        "question": "IHHP 플랜에서 치과 검진 보장돼?",
        "plan_tier": "IHHP",
        "answer_page": "16",
        "answer_keyword": "Examinations, maximum"
    },
    {
        "question": "IHHP Module 4A 치과 연간 한도가 얼마야?",
        "plan_tier": "IHHP",
        "answer_page": "16",
        "answer_keyword": "Special Dental Treatment"
    },
    {
        "question": "IHHP 플랜에서 임플란트 보장돼?",
        "plan_tier": "IHHP",
        "answer_page": "16",
        "answer_keyword": "Dental implants"
    },
    {
        "question": "IHHP 플랜 GP 상담 비용 한도는?",
        "plan_tier": "IHHP",
        "answer_page": "13",
        "answer_keyword": "GP consultations"
    },
    {
        "question": "IHHP 플랜에서 암 치료는 어떻게 돼?",
        "plan_tier": "IHHP",
        "answer_page": "10",
        "answer_keyword": "Cancer treatment"
    },
]

print(f"✅ 총 {len(eval_dataset)}개 평가셋 생성 완료")
print("\n샘플 확인:")
for q in eval_dataset[:3]:
    print(f"  [{q['plan_tier']}] {q['question']}")
    print(f"         → p.{q['answer_page']} | 키워드: {q['answer_keyword']}")

✅ 총 20개 평가셋 생성 완료

샘플 확인:
  [Premier] Premier 플랜에서 처방약 한도가 얼마야?
         → p.19 | 키워드: PRESCRIBED MEDICINES AND DRESSINGS
  [Premier] Premier 플랜 전체 연간 한도는?
         → p.18 | 키워드: Overall annual policy maximum
  [Premier] Premier 플랜에서 암 치료 보장돼?
         → p.18 | 키워드: cancer treatment


In [17]:
import math
from langchain_openai import ChatOpenAI
import json, re

def evaluate_retriever(eval_dataset, vectorstore, k=5):
    
    analyzer = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    hit_count  = 0
    rr_scores  = []
    ndcg_scores = []
    fail_cases = []

    print(f"{'='*65}")
    print(f"🧪 Retriever 평가 시작 (총 {len(eval_dataset)}개 / k={k})")
    print(f"{'='*65}\n")

    for i, case in enumerate(eval_dataset, 1):
        question  = case['question']
        plan      = case['plan_tier']
        ans_page  = str(case['answer_page'])
        ans_kw    = case['answer_keyword'].lower()

        # analyze_node와 동일한 프롬프트
        prompt = f"""You are an insurance query analyzer.
Return STRICT JSON:
{{
  "section_type": "benefit_table|exclusion|claim_process|pre_auth|etc",
  "english_query": "<search query here>"
}}

Rules for section_type:
- benefit_table: 보장 여부, 한도, 금액 관련 → 애매하면 benefit_table
- exclusion: 명시적 제외 항목 질문
- claim_process: 청구 방법/서류
- pre_auth: 사전승인

Rules for english_query:
- Use EXACT terminology from insurance benefit documents
- Good: "prescribed medicines dressings benefit limit GBP"
- Bad: "is cold medicine covered"

Question: {question}"""

        raw      = analyzer.invoke(prompt).content
        json_str = re.search(r'\{.*\}', raw, re.DOTALL).group(0)
        analysis = json.loads(json_str)

        query   = analysis.get("english_query", question)
        section = analysis.get("section_type", "benefit_table")

        # 메타필터 적용
        search_filter = {"$and": [
            {"plan_tier":     plan},
            {"section_type":  section}
        ]}

        docs = vectorstore.similarity_search(query, k=k, filter=search_filter)
        if not docs:
            docs = vectorstore.similarity_search(
                query, k=k, filter={"plan_tier": plan}
            )

        # 정답 판정: 페이지 + 키워드 둘 다 일치해야 정답
        hit  = False
        rank = 0
        retrieved_pages = []

        for r, doc in enumerate(docs, 1):
            page = doc.metadata.get('page', '')
            retrieved_pages.append(page)
            page_match = (page == ans_page)
            kw_match   = (ans_kw in doc.page_content.lower())
            if page_match and kw_match:
                hit  = True
                rank = r
                break

        # 지표 계산
        hit_count += int(hit)
        rr         = 1 / rank if hit else 0
        dcg        = (1 / math.log2(rank + 1)) if hit else 0
        idcg       = 1 / math.log2(2)  # 이상적: 1위에 있을 때
        ndcg       = dcg / idcg

        rr_scores.append(rr)
        ndcg_scores.append(ndcg)

        status = "✅" if hit else "❌"
        rank_str = f"{rank}위" if hit else "-"
        print(f"[{i:2d}] {status} [{plan:12s}] {question}")
        print(f"       쿼리: {query}")
        print(f"       섹션: {section} | 정답 p.{ans_page} | 검색: {retrieved_pages} | {rank_str}")

        if not hit:
            fail_cases.append({
                "question": question,
                "plan":     plan,
                "query":    query,
                "section":  section,
                "ans_page": ans_page,
                "retrieved": retrieved_pages
            })
        print()

    # 최종 지표
    n        = len(eval_dataset)
    hit_rate = hit_count / n
    mrr      = sum(rr_scores)   / n
    ndcg_avg = sum(ndcg_scores) / n

    print(f"{'='*65}")
    print(f"📊 최종 평가 결과 (k={k})")
    print(f"{'='*65}")
    print(f"  Hit Rate @{k} : {hit_rate:.2%}  ({hit_count}/{n})")
    print(f"  MRR          : {mrr:.4f}   (1위=1.0, 2위=0.5, 3위=0.33...)")
    print(f"  NDCG         : {ndcg_avg:.4f}   (순위 가중 적중률)")
    print(f"{'='*65}")

    if fail_cases:
        print(f"\n❌ 미적중 케이스 ({len(fail_cases)}개):")
        for fc in fail_cases:
            print(f"  [{fc['plan']}] {fc['question']}")
            print(f"    쿼리: {fc['query']}")
            print(f"    섹션: {fc['section']} | 정답 p.{fc['ans_page']} | 검색된 페이지: {fc['retrieved']}")

    return {"hit_rate": hit_rate, "mrr": mrr, "ndcg": ndcg_avg}


# 실행
results = evaluate_retriever(eval_dataset, vectorstore, k=5)

🧪 Retriever 평가 시작 (총 20개 / k=5)

[ 1] ✅ [Premier     ] Premier 플랜에서 처방약 한도가 얼마야?
       쿼리: prescribed medicines benefit limit Premier plan
       섹션: benefit_table | 정답 p.19 | 검색: ['18', '18', '17', '19'] | 4위

[ 2] ❌ [Premier     ] Premier 플랜 전체 연간 한도는?
       쿼리: Premier plan overall annual limit
       섹션: benefit_table | 정답 p.18 | 검색: ['18', '18', '20', '19', '18'] | -

[ 3] ✅ [Premier     ] Premier 플랜에서 암 치료 보장돼?
       쿼리: cancer treatment coverage Premier plan
       섹션: benefit_table | 정답 p.18 | 검색: ['18'] | 1위

[ 4] ✅ [Premier     ] Premier 플랜 외래 진료 연간 한도가 얼마야?
       쿼리: Premier plan outpatient treatment annual limit
       섹션: benefit_table | 정답 p.18 | 검색: ['18'] | 1위

[ 5] ✅ [Premier     ] Premier 플랜에서 치과 치료 보장돼?
       쿼리: dental treatment coverage Premier plan
       섹션: benefit_table | 정답 p.20 | 검색: ['18', '20'] | 2위

[ 6] ❌ [Premier     ] Premier 플랜에서 사전승인 필요한 항목이 뭐야?
       쿼리: items requiring pre-authorization Premier plan
       섹션: pre_auth | 정답 p.18 | 검색: ['36', '